# SF3D base mesh

The one 3D lane, run on a Kaggle GPU. The weights arrive as a private dataset, so no token is in the cloud. The model is patched to float16 because the Kaggle T4 has no bf16.

In [ ]:
import os, subprocess, sys, pathlib, json

REPO = "/kaggle/working/sf3d"
WEIGHTS = "/kaggle/input/sf3d-weights"
IN_DIR = "/kaggle/input/concepts"
OUT_DIR = "/kaggle/working/out"

subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Stability-AI/stable-fast-3d", REPO], check=True)

# The two C++ extensions need torch at build time: no build isolation.
os.environ["USE_NATIVE_ARCH"] = "0"
os.environ["CPPFLAGS"] = "-include utility"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--no-build-isolation", f"{REPO}/texture_baker", f"{REPO}/uv_unwrapper"], check=True)

# bfloat16 is not supported on the T4.
run_py = pathlib.Path(REPO, "run.py")
text = run_py.read_text()
run_py.write_text(text.replace("torch.bfloat16", "torch.float16"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "einops==0.7.0", "jaxtyping==0.2.31", "omegaconf==2.3.0", "transformers==4.42.3",
                "open_clip_torch==2.24.0", "trimesh==4.4.1", "numpy==1.26.4", "huggingface-hub==0.23.4",
                "rembg==2.0.57", "gpytoolbox==0.2.0", "pynanoinstantmeshes==0.0.3",
                "gradio==4.41.0", "gradio-litmodel3d==0.0.1"], check=True)
print("deps ready")

In [ ]:
import pathlib, json
# The weights are a private dataset: one safetensors file plus the config.
cfg = next(pathlib.Path(WEIGHTS).glob("*.yaml"))
ckpt = next(pathlib.Path(WEIGHTS).glob("*.safetensors"))
print("weights", cfg.name, ckpt.name, ckpt.stat().st_size)

In [ ]:
import subprocess, sys, os, pathlib, json
imgs = sorted(str(p) for p in pathlib.Path(IN_DIR).glob("*.png")) + sorted(str(p) for p in pathlib.Path(IN_DIR).glob("*.jpg"))
print("images", imgs)
cmd = [sys.executable, "run.py", *imgs, "--output-dir", OUT_DIR,
       "--texture-resolution", "1024", "--foreground-ratio", "0.85",
       "--remesh_option", "none", "--batch_size", "1", "--device", "cuda",
       "--pretrained-model", str(ckpt)]
r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-3000:])
glbs = [str(p) for p in pathlib.Path(OUT_DIR).rglob("*.glb")]
print("GLBS", glbs)
for g in glbs:
    print("GLSIZE", g, os.path.getsize(g))